# Statistics

## Sources and Interpretation

The charts and supporting tables on this page are **generated by notebook code
during book compilation**, using `milestone1/data/reported_results.json`.
This file transcribes Section 5 of the updated `Hieu.docx` and `Dat.docx`,
reviewed on **9 November 2026**. Totals, rates and plotted values are computed
from those aggregates. The original run exports remain pending reconciliation;
these are **document-reported results**, not independently verified raw records.

Each document reports five ranges of 5,000 publications. A publication is a
base arXiv ID, not a version. arXiv source outcomes and Semantic Scholar
reference outcomes describe different stages and must not be combined into a
single success rate. Small failure categories are shown separately so that
they remain visible beside the much larger successful counts.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

from milestone1.reporting import COLORS, chart_style, finish_figure
from matplotlib.ticker import FuncFormatter

plt = chart_style()
labels = [f"Range {r['range']}" for r in arxiv]
positions = list(range(len(labels)))
comma = FuncFormatter(lambda value, _: f"{value:,.0f}")

## arXiv Publication Outcomes

**Figure 1.** Final publication statuses after Fast Crawl and Recovery.
The left panel shows all 5,000 assigned IDs per range. The right panel shows
only non-`ok` records on a separate count scale; it is not an additional group
of publications. Labels in the left panel give `ok` counts and percentages of
assigned IDs. No `not_found` records were reported.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained",
                         gridspec_kw={"width_ratios": [1.35, 1]})
ok = [r["ok"] for r in arxiv]
no_tex = [r["no_tex"] for r in arxiv]
errors = [r["error"] for r in arxiv]
axes[0].barh(positions, ok, color=COLORS["teal"], label="ok")
axes[0].barh(positions, no_tex, left=ok, color=COLORS["orange"], label="no_tex")
axes[0].barh(positions, errors, left=[a+b for a,b in zip(ok, no_tex)],
             color=COLORS["red"], label="error")
for i, r in enumerate(arxiv):
    axes[0].text(r["ok"] / 2, i, f"{r['ok']:,} ({100*r['ok']/r['assigned']:.2f}%)",
                 ha="center", va="center", color="white", fontweight="bold")
axes[0].set(xlim=(0, 5000), title="All assigned publications", xlabel="Publication count")
axes[1].barh(positions, no_tex, color=COLORS["orange"])
axes[1].barh(positions, errors, left=no_tex, color=COLORS["red"], hatch="///")
for i, (n, e) in enumerate(zip(no_tex, errors)):
    axes[1].text(n/2, i, str(n), ha="center", va="center")
    axes[1].text(n+e+6, i, f"{e} errors", va="center", fontsize=10)
axes[1].set(xlim=(0, 400), title="Detail: non-ok publications", xlabel="Publication count (different scale)")
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(comma)
    ax.grid(axis="x", alpha=0.18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=3, frameon=False)
finish_figure(fig, "statistics-arxiv-outcomes",
              "arXiv outcomes by range, with a separate detail panel for no_tex and error records.")

The document reports metadata for all 25,000 IDs across 500 checkpoint batches.
There are 23,695 `ok` publications (94.78%), 22 `error` (0.088%) and 1,283
`no_tex` (5.132%), with no final `not_found` records. The `ok` rate ranges from
93.68% in Range 4 to 96.72% in Range 1, a difference of 3.04 percentage points.

These labels follow the reviewed code: a paper is `error` if any version is
`error`; otherwise it is `ok` if at least one version retained `.tex` or `.bib`,
and `no_tex` otherwise. A successful paper can still contain `no_tex` versions,
and a retained `.bib` alone can satisfy the current `ok` condition. These publication-level counts
do not establish that every version has usable LaTeX.

`no_tex` accounts for about 98.31% of non-`ok` publications. It includes PDF,
empty source, no retained files and source HTTP 404/410. It cannot all be
attributed to intrinsically PDF-only submissions without examining reasons.
The 22 remaining errors can include download, format, extraction or resource
limit failures. Version counts and reason breakdowns remain pending.

## Semantic Scholar Outcomes and Cache Dependence

**Figure 2.** Composition of the supplied rerun. The left panel separates
accepted caches from fresh successful lookups and fresh `not_indexed` results.
The right panel expands the uncached subset on a different count scale.
There are no reported `failed` or `invalid_id` outcomes in this snapshot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained",
                         gridspec_kw={"width_ratios": [1.35, 1]})
cached = [r["cache_hits"] for r in s2]
fresh_ok = [r["ok"] - r["cache_hits"] for r in s2]
missing = [r["not_indexed"] for r in s2]
axes[0].barh(positions, cached, color=COLORS["blue"], label="Accepted cache")
axes[0].barh(positions, fresh_ok, left=cached, color=COLORS["teal"], label="Fresh ok")
axes[0].barh(positions, missing, left=[a+b for a,b in zip(cached, fresh_ok)],
             color=COLORS["red"], label="Fresh not_indexed")
for i, r in enumerate(s2):
    axes[0].text(r["cache_hits"]/2, i, f"{r['cache_hits']:,} cached",
                 ha="center", va="center", color="white", fontweight="bold")
axes[0].set(xlim=(0, 5000), title="All input publications", xlabel="Publication count")
axes[1].barh(positions, fresh_ok, color=COLORS["teal"])
axes[1].barh(positions, missing, left=fresh_ok, color=COLORS["red"], hatch="///")
for i, (n, m) in enumerate(zip(fresh_ok, missing)):
    if n:
        axes[1].text(n/2, i, str(n), ha="center", va="center", color="white", fontweight="bold")
    axes[1].text(n+m+1.5, i, f"{m} not indexed", va="center", fontsize=9)
axes[1].set(xlim=(0, 100), title="Detail: fresh lookups only", xlabel="Publication count (different scale)")
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(comma)
    ax.grid(axis="x", alpha=0.18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=3, frameon=False)
finish_figure(fig, "statistics-semantic-scholar-outcomes",
              "Semantic Scholar rerun outcomes are dominated by caches; fresh lookups are shown separately.")

There are 24,980 `ok` records, 20 HTTP-404 `not_indexed` records and no reported
`failed` or `invalid_id` records in this snapshot. The aggregate status rate is
24,980 / 25,000 = **99.92%**. Of the inputs, 24,811 (99.244%) are cache hits;
only 189 are uncached valid-publication lookups. The 24,980 `ok` count therefore
does not mean 24,980 successful fresh API requests during this rerun. In this
snapshot, subtracting the reported cache hits leaves 169 fresh `ok` outcomes
and 20 `not_indexed` outcomes. A lookup may contain multiple HTTP attempts.

The zero-reference total is 1,881. These are empty records accepted by the
implementation. The current payload check can also accept a missing reference
field as empty, so the number is not independently verified absence of
citations. Likewise, zero final failure flags does not prove no HTTP 429 or
temporary network errors occurred earlier or inside retries.

## Reference Volume and arXiv Coverage

**Figure 3.** Local reference occurrences, split by whether an arXiv ID is
available. The left panel compares counts; the right compares the arXiv-bearing
percentage, using each range's total references as denominator. The dashed line
shows the overall share weighted by reference counts, not an unweighted mean
of the five percentages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained",
                         gridspec_kw={"width_ratios": [1.35, 1]})
with_id = [r["refs_with_arxiv"] for r in s2]
without_id = [r["refs_total"]-r["refs_with_arxiv"] for r in s2]
axes[0].barh(positions, with_id, color=COLORS["teal"], label="With arXiv ID")
axes[0].barh(positions, without_id, left=with_id, color=COLORS["gray"], label="Without arXiv ID")
for i, r in enumerate(s2):
    axes[0].text(r["refs_total"]+2500, i, integer(r["refs_total"]), va="center", fontsize=9)
axes[0].set(xlim=(0, 230000), title="Local reference counts", xlabel="Reference occurrences")
axes[0].xaxis.set_major_formatter(comma)
shares = [100*r["refs_with_arxiv"]/r["refs_total"] for r in s2]
bars = axes[1].barh(positions, shares, color=COLORS["teal"])
axes[1].bar_label(bars, labels=[f"{v:.2f}%" for v in shares], padding=4)
overall_share = 100*total(s2, "refs_with_arxiv")/total(s2, "refs_total")
axes[1].axvline(overall_share, color=COLORS["purple"], linestyle="--", linewidth=1.6,
                label=f"Overall: {overall_share:.2f}%")
axes[1].set(xlim=(0, 100), title="arXiv ID coverage", xlabel="Share of local references (%)")
axes[1].legend(loc="lower right", frameon=False, fontsize=9)
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=2, frameon=False)
finish_figure(fig, "statistics-reference-coverage",
              "Reference counts and arXiv-ID coverage by range, with a weighted overall coverage of 49.83 percent.")

The documents report **902,317** local reference occurrences. Of these,
**449,590 (49.83%)** carry arXiv IDs, while **452,727 (50.17%)** do not. The
overall average is 36.09 reference occurrences per input publication; using
only the 24,980 `ok` records as denominator gives approximately 36.12.

Range 5 has the largest local count (193,120), while Range 1 has the smallest
(163,950) and the highest arXiv-bearing share (56.21%). Keeping references
without arXiv IDs locally preserves a substantial part of the collected
metadata, even though the published reference pool uses the arXiv-bearing
subset.

These counts are citation occurrences associated with citing publications,
not the number of globally unique cited works. The same work cited by two
different papers contributes two relations. Counts before and after the
identifier-tuple deduplication, and global distinct-work counts, are not
established by the supplied aggregates.

## References per Publication and Empty Records

**Figure 4.** The left panel shows the mean reference count per input
publication; the right shows the percentage of inputs accepted with zero
references. Both use all 5,000 input publications in each range. These are
different metrics with separate axes, not distributions of individual papers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained")
means = [r["refs_total"]/r["input_papers"] for r in s2]
empty = [100*r["zero_refs"]/r["input_papers"] for r in s2]
bars = axes[0].barh(positions, means, color=COLORS["blue"])
axes[0].bar_label(bars, labels=[f"{v:.2f}" for v in means], padding=4)
overall_mean = total(s2, "refs_total")/total(s2, "input_papers")
axes[0].axvline(overall_mean, color=COLORS["purple"], linestyle="--",
                label=f"Overall: {overall_mean:.2f}")
axes[0].set(xlim=(0, 46), title="Mean local reference count", xlabel="Reference occurrences / input publication")
axes[0].legend(loc="lower left", frameon=False, fontsize=9)
bars = axes[1].barh(positions, empty, color=COLORS["orange"])
axes[1].bar_label(bars, labels=[f"{v:.2f}% ({r['zero_refs']:,})" for v,r in zip(empty,s2)], padding=4)
axes[1].set(xlim=(0, 15), title="Accepted zero-reference records", xlabel="Share of input publications (%)")
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.18)
finish_figure(fig, "statistics-references-per-publication",
              "Mean reference counts and zero-reference rates across the five ranges, with all inputs as denominators.")

Range 5 has the highest mean (38.62 references per input publication) and
the lowest zero-reference share (5.62%); Range 1 has the lowest mean (32.79)
and highest zero-reference share (9.60%). These aggregate differences do not
establish a causal relationship or reveal the median, spread or outliers of
per-paper reference counts. Those require the original publication-level records.

Empty accepted records are not proof that the papers have no citations: the
reviewed payload handling can accept a missing reference field as empty.

## Supporting Tables and Validation

The following tables retain exact counts for inspection. Charts above provide
the primary comparison; these tables are also generated during compilation.

**Table 1.** Exact arXiv publication outcomes and success rates.

In [ ]:
keys = ("assigned", "metadata", "ok", "error", "no_tex", "not_found")
rows = [[f"Range {r['range']}", *[integer(r[k]) for k in keys],
         percent(r["ok"], r["assigned"])] for r in arxiv]
rows.append(["Overall", *[integer(total(arxiv, k)) for k in keys],
             percent(total(arxiv, "ok"), total(arxiv, "assigned"))])
table(["Range", "Assigned IDs", "Metadata", "ok", "error", "no_tex", "not_found", "ok / assigned"], rows)

**Table 2.** Exact local reference counts, arXiv-bearing shares and mean counts.

In [ ]:
rows = []
for r in s2:
    rows.append([f"Range {r['range']}", integer(r["refs_total"]), integer(r["refs_with_arxiv"]),
                 integer(r["refs_total"] - r["refs_with_arxiv"]),
                 percent(r["refs_with_arxiv"], r["refs_total"]),
                 f"{r['refs_total'] / r['input_papers']:.2f}"])
all_refs, arxiv_refs, inputs = total(s2, "refs_total"), total(s2, "refs_with_arxiv"), total(s2, "input_papers")
rows.append(["Overall", integer(all_refs), integer(arxiv_refs), integer(all_refs - arxiv_refs),
             percent(arxiv_refs, all_refs), f"{all_refs / inputs:.2f}"])
table(["Range", "All local refs", "With arXiv ID", "Without arXiv ID", "arXiv share", "Refs / input paper"], rows)

**Table 3.** Metadata findings reported by the current validator. The reported
`papers` row count is 5,000 in each range; the reported `references` row count
equals the arXiv-bearing count in Table 2. Ranges 1--3 report no issues;
Ranges 4 and 5 each report one empty author list. Exports are written even
when issues are present.

In [ ]:
details = data["semantic_scholar"]["validation_details"]
table(["Range", "Table", "Reported index", "Field", "Finding"],
      [[f"Range {r['range']}", r["table"], str(r["reported_index"]), r["field"], r["problem"]] for r in details])

Ranges 1--3 report no validator issues. Ranges 4 and 5 each report an empty
`ref_authors` list, at the indices shown above. These are findings under the
implemented non-empty-author validation rule; their source records and the
agreed missing-metadata policy are pending review. No missing authors are
invented and no records are removed merely to force a passing result.

The validator checks selected columns, types, emptiness and duplicate keys.
It does not establish complete bibliographic accuracy, all date formats or all
cross-table relationships. A cited `ref_arxiv_id` is allowed to be outside the
assigned `papers` corpus.

## Remaining Evidence

Reconciliation requires the original per-range `summary.json`, `metrics.csv`,
`runtime_history.csv`, `crawl_records.csv` and `failed_versions.csv` for arXiv,
and `metrics_s2_references.json`, `s2_crawl_records.csv`,
`validation_report.json` for Semantic Scholar. Version coverage, error/no-TeX
reasons and the two author issues can then be checked against records.

The reported aggregates above remain usable as attributed current results while
those exports and the requested corrections are pending. Neither a fresh crawl
nor any external API request is performed by this notebook.